Description:

Analyze the relationship between the number of primary care appointments per head of population in sub-ICBs within SNEE (Suffolk and North East Essex) and staffing levels per head of population. The goal is to use the GP Patient list, the most recent appointments dataset, and the staffing dataset from NHSE (all currently available in the catalog) to investigate how the patient-to-GP ratio correlates with appointments per head specifically in the context of SNEE compared to broader English sub-ICBs.

Exam question/Objective:

Determine if staffing levels per head of population significantly affect the number of primary care appointments per head in sub-ICBs within SNEE. Additionally, assess whether the patient-to-GP ratio in SNEE reflects or deviates from the trends seen in other English sub-ICBs.

Stakeholder Request:

I want a report that details the relationship between the number of primary care appointments per head of population and staffing levels per head of population in sub-ICBs within SNEE. The report should highlight if variations in patient-to-GP ratios contribute to differences in appointments per head and whether these findings are consistent with or diverge from other English sub-ICBs.

Key Data Sources:

GP Patient list from NHSE
Most recent appointments dataset from NHSE
Staffing dataset from NHSE
Methodology/Approach:

Data extraction from NHSE catalog for GP Patient list, appointments data, and staffing data.
Clean and preprocess the datasets to ensure consistency.
Calculate patient-to-GP ratios and staffing levels per head of population.
Conduct initial exploratory data analysis (EDA) to understand distribution and correlations.
Apply statistical analysis (e.g., linear regression or correlation analysis) to assess relationships.
Compare results between SNEE sub-ICBs and other English sub-ICBs.
Visualize findings through charts and graphs.
Summarize insights in a report.
Outputs:

Branch in repo
Notebook
Markdown blog

## Library Imports

In [1]:
import os
from pathlib import Path
if 'notebooks' in str(Path.cwd()):
    os.chdir('..')

# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from matplotlib.ticker import FuncFormatter
import datetime as dt
import pickle
from typing import Dict
from time import sleep

# project imports from src
from src.schemas import DataCatalog
from src.various_methods import PlotCounter, get_workingdays
from src import constants

# Importing SNEE styles
from sneeifstyles import mpl_style
mpl_style()

import warnings
warnings.filterwarnings("ignore")

## Initial Set-up

In [ ]:
## Constants
SNEE_SUB_ICB = ['06L','06T','07K']
SNEE_SUB_ICB_NAMES = ['Ipswich & East Suffolk', 'North East Essex', 'West Suffolk']
NOTEBOOK_ALIAS = "Appointments"
GP_PATIENTS_LIST_CATALOG = 'Patients Registered at a GP practice, September 2024'
GP_APPOINTMENTS_CATALOG_NAME:str = 'Appointments in General Practice, August 2024'
GP_WORKFORCE_CATALOG:str = 'General Practice workforce'  # August/2024
GP_LIST_AGE_BANDS = constants.GP_LIST_AGE_BANDS 
GP_LIST_AGE_LABELS = constants.GP_LIST_AGE_LABELS 
SNEE_SUBICB_CODES = list(constants.ONS_CODES.keys())

# Loading the Data Catalog
catalog =  DataCatalog.load_from_yaml("data_catalog.yaml")

# Initializing the plotCounter object
plot_counter = PlotCounter(name=NOTEBOOK_ALIAS)

# set up output directories
for i in ['outputs/assumptions', 'outputs/plots', 'outputs/tables']:
    if not os.path.exists(i):
        os.makedirs(i)

## 1. GP list Loading and processing

In [ ]:
def process_gp_list(df):
    """
    Args:
        df (pandas.DataFrame): GP LIST DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Keeping only the total population both 'AGE_GROUP_5' and 'SEX' columns are 'ALL'
    df_ = df.loc[(df['AGE_GROUP_5'] == 'ALL') & (df['SEX'] == 'ALL')]
    
    # Filter rows to keep only those with ORG_TYPE as 'ICB' or 'SUB_ICB_LOCATION_CODE'
    df_ = df_[df_['ORG_TYPE'].isin(['ICB', 'SUB_ICB_LOCATION_CODE'])].copy()
    
    # Create new columns 'ICB' and 'SUB_ICB_LOCATION_CODE' with values based on 'ORG_TYPE'
    df_['ICB'] = df_.apply(lambda x: x['ORG_CODE'] if x['ORG_TYPE'] == 'ICB' else None, axis=1)
    df_['SUB_ICB_LOCATION_CODE'] = df_.apply(lambda x: x['ORG_CODE'] if x['ORG_TYPE'] == 'SUB_ICB_LOCATION_CODE' else None, axis=1)

    #Dropping unused columns
    df_ = df_.drop(columns=['PUBLICATION','EXTRACT_DATE','ORG_TYPE','POSTCODE','SEX','AGE_GROUP_5'])
  
    return df_

In [196]:
gp_list_df = catalog.get_catalog_entry_by_name(GP_PATIENTS_LIST_CATALOG)
patients_df = gp_list_df.load()
patients_df = process_gp_list(patients_df)

# ICB level total
icb_patients_df = patients_df[~patients_df['ICB'].isnull()].drop(columns={'SUB_ICB_LOCATION_CODE'}).reset_index(drop=True)
icb_patients_df.tail()

,ORG_CODE,ONS_CODE,NUMBER_OF_PATIENTS,ICB
37,QWE,E54000031,1771321,QWE
38,QWO,E54000054,2691174,QWO
39,QWU,E54000018,1111224,QWU
40,QXU,E54000063,1146808,QXU
41,QYG,E54000008,2787872,QYG


In [197]:
# SUB-ICB level total
sub_icb_patients_df = patients_df[~patients_df['SUB_ICB_LOCATION_CODE'].isnull()].drop(columns={'ICB'}).reset_index(drop=True)
sub_icb_patients_df.tail()

,ORG_CODE,ONS_CODE,NUMBER_OF_PATIENTS,SUB_ICB_LOCATION_CODE
101,D9Y0V,E38000253,1730028,D9Y0V
102,M1J4Y,E38000249,1140994,M1J4Y
103,M2L0M,E38000257,534870,M2L0M
104,W2U3Z,E38000256,2899512,W2U3Z
105,X2C4Y,E38000254,461152,X2C4Y


## 2. Workforce Loading and processing

In [191]:
def process_workforce(df):
    """
    Args:
        df (pandas.DataFrame): Workforce DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Dropping unused columns
    df_ = df_.drop(columns=['YEAR','Month','COMM_REGION_CODE','COMM_REGION_NAME','ICB_NAME','SUB_ICB_NAME','DATA_SOURCE','STAFF_ROLE',
                            'UNIQUE_IDENTIFIER','DETAILED_STAFF_ROLE','COUNTRY_QUALIFICATION_AREA','COUNTRY_QUALIFICATION_GROUP','AGE_BAND','AGE_YEARS'])
    df_ = df_.reset_index(drop=True)
    
    # Group by
    df_ = df_.groupby(['ICB_CODE', 'SUB_ICB_CODE', 'STAFF_GROUP', 'GENDER']).agg(
            FTE_mean=('FTE', 'mean'),  # Calculate mean of FTE
            Occurrences=('FTE', 'size')  # Count occurrences in each group
            ).reset_index()

    return df_

In [192]:
workforce_entry =  catalog.get_catalog_entry_by_name(GP_WORKFORCE_CATALOG)
workforce_df = workforce_entry.load()

# To match the names of icbs at the end
col = ['ICB_CODE','ICB_NAME','SUB_ICB_CODE','SUB_ICB_NAME']
icb_name_code_match = workforce_df[col].drop_duplicates().reset_index(drop=True)

workforce_df = process_workforce(workforce_df)
workforce_df

,ICB_CODE,SUB_ICB_CODE,STAFF_GROUP,GENDER,FTE_mean,Occurrences
0,QE1,00Q,Admin/Non-Clinical,Female,0.757583,259
1,QE1,00Q,Admin/Non-Clinical,Male,0.843641,42
2,QE1,00Q,Admin/Non-Clinical,Other/Unknown,0.533333,1
3,QE1,00Q,Direct Patient Care,Female,0.730179,26
4,QE1,00Q,Direct Patient Care,Male,0.909333,5
...,...,...,...,...,...,...
1102,QYG,99A,Nurses,Male,0.861333,15
1103,QYG,99A,Nurses,Other/Unknown,0.698319,21
1104,Unknown,Unknown,GP,Female,1.020044,453
1105,Unknown,Unknown,GP,Male,1.047309,301


## 3. Appointments Loading and processing

In [ ]:
def process_appointments(df):
    """
    Args:
        df (pandas.DataFrame): Appointments DataFrame. 
    Returns:
        pandas.DataFrame: 
    """
    df_ = df.copy()
    
    # Convert the 'date_column' to datetime format
    df_['Appointment_Date'] = pd.to_datetime(df_['Appointment_Date'], format='%d%b%Y')
    
    # Filter the DataFrame to keep only rows from the latest month
    latest_month = df_['Appointment_Date'].dt.to_period('M').max()
    df_ = df_[df_['Appointment_Date'].dt.to_period('M') == latest_month].reset_index(drop=True)
    
    # Dropping unused columns
    df_ = df_.drop(columns=['SUB_ICB_LOCATION_NAME','SUB_ICB_LOCATION_CODE','REGION_ONS_CODE','ACTUAL_DURATION'])
    
    # Group by 'ICB_ONS_CODE', 'SUB_ICB_LOCATION_ONS_CODE', and sum 'COUNT_OF_APPOINTMENTS' across all months
    df_ = df_.groupby(['ICB_ONS_CODE', 'SUB_ICB_LOCATION_ONS_CODE'])['COUNT_OF_APPOINTMENTS'].sum().reset_index()

    return df_

In [198]:
appointments_catalog_entry = catalog.get_catalog_entry_by_name(GP_APPOINTMENTS_CATALOG_NAME)
appointments_df = appointments_catalog_entry.load()
appointments_df = process_appointments(appointments_df)
appointments_df

,ICB_ONS_CODE,SUB_ICB_LOCATION_ONS_CODE,COUNT_OF_APPOINTMENTS
0,E54000008,E38000068,44866
1,E54000008,E38000091,66466
2,E54000008,E38000101,208663
3,E54000008,E38000161,45549
4,E54000008,E38000170,59090
...,...,...,...
101,E54000061,E38000006,113107
102,E54000061,E38000044,142867
103,E54000061,E38000141,131454
104,E54000061,E38000146,265247


In [182]:
for col in appointments_df.columns:
    null_count = appointments_df[col].nunique()
    zero_count = (appointments_df[col] == 0).sum()
    print(f'{col} - NaN count: {null_count}, Zero count: {zero_count}')

ICB_ONS_CODE - NaN count: 42, Zero count: 0
SUB_ICB_LOCATION_ONS_CODE - NaN count: 106, Zero count: 0
COUNT_OF_APPOINTMENTS - NaN count: 106, Zero count: 0
